# Notebook 02 — Diseño del detector

Este notebook implementa las cuatro capas del detector de cierre de válvulas: sanidad (Capa 0), suavizado (Capa 1), regresión P-Q (Capa 2) y CUSUM (Capa 3). Las funciones se prototipan inline aquí y se refactorizan a `src/` una vez que se estabilizan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Carga del dataset

In [ ]:
DATA_PATH = Path("..") / "data" / "PLOT_TS_P_Q.csv"

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    decimal=".",
    parse_dates=["TS"],
    date_format="%d/%m/%Y %H:%M:%S",
)

print(df.shape)
df.head(3)

## 2. Capa 0 — Sanidad

La regla descarta las filas donde `Q == 0` y `P > 50`. Estos cuatro registros del 12-abr (00:07–00:10) son glitches físicamente imposibles identificados en el notebook 01: si el gasoducto está presurizado (P ≈ 75), no puede haber caudal nulo.

La asimetría de la condición es deliberada: `Q = 0` con `P` alta es imposible físicamente y siempre indica error de telemetría. `Q = 0` con `P` baja, en cambio, podría corresponder a planta parada (estado legítimo) y no debe filtrarse. El umbral de 50 está bien por debajo del mínimo operativo observado (~65), con margen suficiente para no capturar ningún estado real de baja presión.

In [ ]:
def aplicar_sanidad(df: pd.DataFrame, q_max: float = 0.0,
                    p_min: float = 50.0) -> pd.DataFrame:
    """Descarta filas con Q <= q_max y P > p_min.

    Elimina glitches físicamente imposibles: caudal nulo con sistema
    presurizado. Los defaults (q_max=0.0, p_min=50.0) corresponden al
    caso validado en el notebook 01 sobre este dataset.
    Devuelve un DataFrame nuevo con índice reseteado.
    """
    mascara_glitch = (df["Q"] <= q_max) & (df["P"] > p_min)
    return df[~mascara_glitch].reset_index(drop=True)

In [ ]:
df_sano = aplicar_sanidad(df)

print(f"df      : {df.shape}")
print(f"df_sano : {df_sano.shape}")

## 2.1 Validación de la Capa 0

Tres verificaciones automáticas:
1. Que se filtraron exactamente 4 filas.
2. Que las filas filtradas corresponden a los timestamps del 12-abr identificados en el notebook 01.
3. Que la condición `Q == 0` y `P > 50` ya no aparece en `df_sano`.

In [ ]:
# 1. Cantidad de filas filtradas
n_filtradas = df.shape[0] - df_sano.shape[0]
assert n_filtradas == 4, f"Se esperaban 4 filas filtradas, se obtuvieron {n_filtradas}"
print(f"Filas filtradas: {n_filtradas}")

# 2. Timestamps de las filas filtradas
mask_filtradas = ~df["TS"].isin(df_sano["TS"])
ts_filtradas = sorted(df.loc[mask_filtradas, "TS"].tolist())
ts_esperados = [
    pd.Timestamp("2026-04-12 00:07:00"),
    pd.Timestamp("2026-04-12 00:08:00"),
    pd.Timestamp("2026-04-12 00:09:00"),
    pd.Timestamp("2026-04-12 00:10:00"),
]
assert ts_filtradas == ts_esperados, f"Timestamps incorrectos: {ts_filtradas}"
print(f"Timestamps filtrados: {[str(ts) for ts in ts_filtradas]}")

# 3. Ausencia de la condicion en la salida
n_residual = ((df_sano["Q"] == 0) & (df_sano["P"] > 50)).sum()
assert n_residual == 0, f"Quedan {n_residual} filas con Q=0 y P>50 en df_sano"
print(f"Filas con Q=0 y P>50 en df_sano: {n_residual}")

print("\u2713 Capa 0 validada")